# Fase 0 — Preparação de Dados: Candidatos 2026 (TSE)

Este notebook baixa e prepara os dados de candidatos das Eleições 2026, e salva um **dataset limpo** que todas as fases seguintes (1 a 4) vão reutilizar — assim ninguém precisa baixar e limpar os dados de novo em cada notebook.

## Configuração

Ajuste as três variáveis abaixo antes de rodar. É a **única coisa que muda** entre a análise do professor (Brasil, Governador) e o trabalho de vocês (um cargo, uma UF).

In [ ]:
CARGO = "DEPUTADO FEDERAL"   # ex.: "GOVERNADOR", "SENADOR", "DEPUTADO FEDERAL"
UF = "SP", "RJ", "MG", "ES"               # None = Brasil inteiro; ou a sigla da UF, ex.: "SP", "BA", "MG"
ANO_ELEICAO = 2026

> **Para os alunos:** troquem `CARGO` para `"SENADOR"` ou `"DEPUTADO FEDERAL"` e `UF` para a sigla do estado do seu trabalho (ex.: `UF = "PE"`). O resto do notebook não muda.

> **Sobre "congelar" a versão dos dados:** o arquivo do TSE pode ser atualizado por eles a qualquer momento (novas candidaturas, correções, impugnações). Para não ficar refém disso, os zips baixados manualmente ficam parados em `dados/` e só mudam quando você baixar uma versão nova de propósito. O que fica **versionado no git** é o resultado desta fase (`dados/*.csv`) — depois de rodar e conferir, faça `git commit` desse CSV: esse commit *é* a versão congelada que a turma toda vai usar.

## 1) Preparação do ambiente

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

os.makedirs('dados', exist_ok=True)

## 2) Obter os dados

O download automático pelo CDN do TSE é bloqueado por um firewall (Akamai) que rejeita requisições que não vêm de um navegador de verdade — isso acontece tanto no Colab quanto localmente, então não vale a pena automatizar. **O fluxo é manual:**

1. Baixe os dois arquivos pelo navegador:
   - Candidatos: `https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand/consulta_cand_2026.zip`
   - Bens: `https://cdn.tse.jus.br/estatistica/sead/odsele/bem_candidato/bem_candidato_2026.zip`
2. Salve os dois **dentro da pasta `dados/`**, com esses nomes exatos:
   - `dados/consulta_cand_2026.zip`
   - `dados/bem_candidato_2026.zip`
3. Rode as células abaixo — elas conferem se os arquivos estão no lugar certo e descompactam.

In [ ]:
ZIP_CAND = f"dados/consulta_cand_{2026}.zip"
ZIP_BEM = f"dados/bem_candidato_{2026}.zip"
#ZIP_CONTAS = f"dados/prestacao_de_contas_eleitorais_candidatos_{2026}.zip"

def conferir_zip(caminho):
    if not os.path.exists(caminho):
        raise FileNotFoundError(
            f"Não encontrei {caminho}. Baixe o arquivo pelo navegador e salve exatamente nesse caminho "
            "(veja os links na célula de markdown acima)."
        )
    if not zipfile.is_zipfile(caminho):
        raise ValueError(f"{caminho} existe, mas não é um .zip válido — baixe de novo pelo navegador.")
    tamanho_mb = os.path.getsize(caminho) / (1024 * 1024)
    print(f"OK: {caminho} ({tamanho_mb:.1f} MB)")

def descompactar(nome_arquivo_zip, destino):
    with zipfile.ZipFile(nome_arquivo_zip, 'r') as zip_ref:
        zip_ref.extractall(destino)
    print(f"Descompactado em {destino}")

conferir_zip(ZIP_CAND)
conferir_zip(ZIP_BEM)
#conferir_zip(ZIP_CONTAS)

OK: dados/consulta_cand_2026.zip (3.0 MB)
OK: dados/bem_candidato_2026.zip (3.6 MB)


In [ ]:
descompactar(ZIP_CAND, './files_cand')
descompactar(ZIP_BEM, './files_bem')
#descompactar(ZIP_CONTAS, './files_contas')

Descompactado em ./files_cand
Descompactado em ./files_bem


## 3) Filtrar candidatos (CARGO + UF)

In [ ]:
df = pd.read_csv(f'./files_cand/consulta_cand_{ANO_ELEICAO}_BRASIL.csv', sep=';', encoding='latin-1')

print(f"Extrato gerado pelo TSE em: {df['DT_GERACAO'].iloc[0]} {df['HH_GERACAO'].iloc[0]}")

# [ALTERAÇÃO] filtro por UF com .isin (funciona com None, string ou lista/tupla)
if UF is not None:
    ufs = list(UF) if isinstance(UF, (list, tuple)) else [UF]
    df_filtrado = df[df['SG_UF'].isin(ufs)]
else:
    df_filtrado = df.copy()

print("\nCargos disponíveis:")
print(df_filtrado.DS_CARGO.value_counts())

dfCargo = df_filtrado[df_filtrado['DS_CARGO'] == CARGO].copy()  # [ALTERAÇÃO] filtrar em df_filtrado, não em df

dfCargoFinal = dfCargo.loc[dfCargo.groupby('SQ_CANDIDATO')['NR_TURNO'].idxmax()]

print()
print(f"CARGO={CARGO!r}  UF={UF!r}  ->  {dfCargoFinal.shape[0]} candidatos")
dfCargoFinal[['SQ_CANDIDATO', 'NM_CANDIDATO', 'SG_UF', 'SG_PARTIDO', 'DT_NASCIMENTO']].head()

Extrato gerado pelo TSE em: 23/08/2026 19:30:42

Cargos disponíveis:
DS_CARGO
DEPUTADO ESTADUAL    3994
DEPUTADO FEDERAL     2793
1º SUPLENTE            63
2º SUPLENTE            60
SENADOR                60
VICE-GOVERNADOR        33
GOVERNADOR             32
Name: count, dtype: int64

CARGO='DEPUTADO FEDERAL'  UF=('SP', 'RJ', 'MG', 'ES')  ->  2793 candidatos


,SQ_CANDIDATO,NM_CANDIDATO,SG_UF,SG_PARTIDO,DT_NASCIMENTO
7137,80002531643,JOÃO BATISTA DE ASSIS,ES,NOVO,23/03/1974
1986,80002531644,FABÍOLA SAQUETTO DI TOMMASO,ES,NOVO,06/08/1977
17538,80002531645,AILANA REGINA PELISSARI SIMONELLI MOREIRA,ES,NOVO,28/11/1978
9686,80002531646,JÚLIO CÉSAR COSTA CAIADO,ES,NOVO,17/04/1959
17539,80002531647,LEONARDO RODRIGO NASCIMENTO,ES,NOVO,14/11/1979


## 4) Patrimônio declarado (bens)

In [ ]:
dfBem = pd.read_csv('./files_bem/bem_candidato_%d_BRASIL.csv' % ANO_ELEICAO, sep=';', encoding='latin-1')
dfBem['VR_BEM_CANDIDATO'] = dfBem['VR_BEM_CANDIDATO'].str.replace(',', '.').astype(float)

def categorize_assets(asset):
    a = asset.lower()
    if 'veículo' in a or 'moto' in a or 'aeronave' in a or 'embarcação' in a:
        return 'BENS MÓVEIS'
    elif 'casa' in a or 'terreno' in a or 'apartamento' in a or 'prédio' in a or 'loja' in a or 'terra nua' in a:
        return 'BENS IMÓVEIS'
    elif 'poupança' in a or 'renda fixa' in a or 'cdb' in a or 'rdb' in a:
        return 'INVESTIMENTOS RENDA FIXA'
    elif 'ações' in a or 'fundo' in a or 'investimento' in a or 'mercado' in a:
        return 'INVESTIMENTOS RENDA VARIÁVEL'
    elif 'dinheiro' in a or 'depósito bancário' in a or 'numerário' in a:
        return 'DINHEIRO'
    else:
        return 'OUTROS BENS'

dfBem['Categoria_BEM'] = dfBem['DS_TIPO_BEM_CANDIDATO'].apply(categorize_assets)

dfBem_pivot = dfBem.pivot_table(
    index='SQ_CANDIDATO', columns='Categoria_BEM', values='VR_BEM_CANDIDATO', aggfunc='sum'
)
dfBem_pivot.fillna(0, inplace=True)
dfBem_pivot['Total_Bens'] = dfBem_pivot.sum(axis=1)
dfBem_pivot['Total_Bens_Log'] = np.log1p(dfBem_pivot['Total_Bens'])

dfBem_pivot[['Total_Bens', 'Total_Bens_Log']].describe()

Categoria_BEM,Total_Bens,Total_Bens_Log
count,1.373000e+04,13730.000000
mean,2.496684e+06,12.693722
std,2.771848e+07,2.044055
min,0.000000e+00,0.000000
25%,1.165086e+05,11.665729
50%,4.014611e+05,12.902868
75%,1.130000e+06,13.937729
max,2.075000e+09,21.453227


In [ ]:
# ============================================================
# [ALTERAÇÃO] DESPESAS DE CAMPANHA (contratadas, já filtradas)
# Arquivo gerado no notebook 05 / preparação de despesas
# ============================================================

# Caminho do CSV que você exportou (ajuste se o nome/pasta for outro)
ARQUIVO_GASTOS = "dados/despesas_Deputado_Federal_('SP', 'RJ', 'MG', 'ES')_2026.csv"
# Alternativa, se estiver na pasta do Colab:
# ARQUIVO_GASTOS = "despesas_Deputado_Federal_('SP', 'RJ', 'MG', 'ES')_2026.csv"

df_desp = pd.read_csv(ARQUIVO_GASTOS)

# Garante número
df_desp["VR_DESPESA_CONTRATADA"] = pd.to_numeric(
    df_desp["VR_DESPESA_CONTRATADA"], errors="coerce"
).fillna(0)

# Total por candidato (várias linhas de origem → uma linha)
gastos = (
    df_desp
    .groupby("SQ_CANDIDATO", as_index=False)["VR_DESPESA_CONTRATADA"]
    .sum()
    .rename(columns={"VR_DESPESA_CONTRATADA": "Total_Gasto_Campanha"})
)
gastos["Total_Gasto_Campanha_Log"] = np.log1p(gastos["Total_Gasto_Campanha"])

print(f"Candidatos com despesa: {len(gastos)}")
print(
    "Interseção com candidatos filtrados:",
    len(set(gastos["SQ_CANDIDATO"]) & set(dfCargoFinal["SQ_CANDIDATO"]))
)
print(gastos.head())

Candidatos com despesa: 2033
Interseção com candidatos filtrados: 2028
   SQ_CANDIDATO  Total_Gasto_Campanha  Total_Gasto_Campanha_Log
0   80002531644               3750.00                  8.229778
1   80002531645               5000.00                  8.517393
2   80002531646                  9.95                  2.393339
3   80002531647               5275.19                  8.570960
4   80002531648               8985.00                  9.103423


In [ ]:
print(
    "Interseção com candidatos filtrados:",
    len(set(gastos["SQ_CANDIDATO"]) & set(dfCargoFinal["SQ_CANDIDATO"]))
)

Interseção com candidatos filtrados: 2028


## 5) Juntar tudo, calcular idade e limpar

A idade mínima elegível **depende do cargo** (regra constitucional) — por isso o filtro de idade é montado a partir de `CARGO`, não fixo.

In [ ]:
# Bloco 8: Unificação dos dados, cálculo de idade e inclusão dos gastos (com alterações)
IDADE_MINIMA_POR_CARGO = {
    'PRESIDENTE': 35, 'VICE-PRESIDENTE': 35,
    'GOVERNADOR': 30, 'VICE-GOVERNADOR': 30,
    'SENADOR': 35,
    'DEPUTADO FEDERAL': 21, 'DEPUTADO ESTADUAL': 21, 'DEPUTADO DISTRITAL': 21,
    'PREFEITO': 21, 'VICE-PREFEITO': 21,
    'VEREADOR': 18,
}
idade_minima = IDADE_MINIMA_POR_CARGO.get(CARGO, 18)
print(f"Idade mínima elegível para {CARGO}: {idade_minima} anos")

dfCandBem = dfCargoFinal.merge(dfBem_pivot, on='SQ_CANDIDATO', how='left')

# [ALTERAÇÃO]: Realização do merge do DataFrame de gastos de campanha com dfCandBem
dfCandBem = dfCandBem.merge(gastos, on='SQ_CANDIDATO', how='left')

# [ALTERAÇÃO]: Tratamento dos valores nulos resultantes do merge dos gastos de campanha
dfCandBem['Total_Gasto_Campanha'] = dfCandBem['Total_Gasto_Campanha'].fillna(0)
dfCandBem['Total_Gasto_Campanha_Log'] = dfCandBem['Total_Gasto_Campanha_Log'].fillna(0)

dfCandBem['Total_Bens'] = dfCandBem['Total_Bens'].fillna(0)
dfCandBem['Total_Bens_Log'] = dfCandBem['Total_Bens_Log'].fillna(0)

dfCandBem['DT_ELEICAO'] = pd.to_datetime(dfCandBem['DT_ELEICAO'], format='%d/%m/%Y')
dfCandBem['DT_NASCIMENTO'] = pd.to_datetime(dfCandBem['DT_NASCIMENTO'], format='%d/%m/%Y')
dfCandBem['IDADE'] = (dfCandBem['DT_ELEICAO'] - dfCandBem['DT_NASCIMENTO']).dt.days // 365

antes = dfCandBem.shape[0]
dfCandBem = dfCandBem[dfCandBem['IDADE'].between(idade_minima, 100)].copy()
print(f"Candidatos removidos por idade inválida: {antes - dfCandBem.shape[0]}")

print(dfCandBem.shape)

# [ALTERAÇÃO]: Exibição de amostragem contendo as colunas de gastos de campanha adicionadas
dfCandBem[['NM_CANDIDATO', 'SG_UF', 'IDADE', 'Total_Bens', 'Total_Gasto_Campanha', 'Total_Gasto_Campanha_Log']].sample(min(5, len(dfCandBem)))

# ============================================================
# UNIFICAÇÃO DOS DADOS (MERGE FINAL DA FASE 0)
# ============================================================

df_final = dfCargoFinal.merge(
    dfBem_pivot,
    left_on="SQ_CANDIDATO",
    right_index=True,
    how="left"
)

df_final = df_final.merge(
    gastos,
    on="SQ_CANDIDATO",
    how="left"
)

colunas_bens = list(dfBem_pivot.columns)
for col in colunas_bens:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna(0)

df_final["Total_Gasto_Campanha"] = df_final["Total_Gasto_Campanha"].fillna(0)
df_final["Total_Gasto_Campanha_Log"] = df_final["Total_Gasto_Campanha_Log"].fillna(0)

print(f"Merge concluído com sucesso! Total de candidatos processados: {len(df_final)}")

Idade mínima elegível para DEPUTADO FEDERAL: 21 anos
Candidatos removidos por idade inválida: 2
(2791, 61)
Merge concluído com sucesso! Total de candidatos processados: 2793


In [ ]:
IDADE_MINIMA_POR_CARGO = {
    'PRESIDENTE': 35, 'VICE-PRESIDENTE': 35,
    'GOVERNADOR': 30, 'VICE-GOVERNADOR': 30,
    'SENADOR': 35,
    'DEPUTADO FEDERAL': 21, 'DEPUTADO ESTADUAL': 21, 'DEPUTADO DISTRITAL': 21,
    'PREFEITO': 21, 'VICE-PREFEITO': 21,
    'VEREADOR': 18,
}
idade_minima = IDADE_MINIMA_POR_CARGO.get(CARGO, 18)
print(f"Idade mínima elegível para {CARGO}: {idade_minima} anos")

dfCandBem = dfCargoFinal.merge(dfBem_pivot, on="SQ_CANDIDATO", how="left")

# left join: quem não tem despesa fica NaN → vira 0
dfCandBem = dfCandBem.merge(gastos, on="SQ_CANDIDATO", how="left")

dfCandBem["Total_Gasto_Campanha"] = dfCandBem["Total_Gasto_Campanha"].fillna(0)
dfCandBem["Total_Gasto_Campanha_Log"] = np.log1p(dfCandBem["Total_Gasto_Campanha"])

dfCandBem["Total_Bens"] = dfCandBem["Total_Bens"].fillna(0)
dfCandBem["Total_Bens_Log"] = dfCandBem["Total_Bens_Log"].fillna(0)

dfCandBem["DT_ELEICAO"] = pd.to_datetime(dfCandBem["DT_ELEICAO"], format="%d/%m/%Y")
dfCandBem["DT_NASCIMENTO"] = pd.to_datetime(dfCandBem["DT_NASCIMENTO"], format="%d/%m/%Y")
dfCandBem["IDADE"] = (dfCandBem["DT_ELEICAO"] - dfCandBem["DT_NASCIMENTO"]).dt.days // 365

antes = dfCandBem.shape[0]
dfCandBem = dfCandBem[dfCandBem["IDADE"].between(idade_minima, 100)].copy()
print(f"Candidatos removidos por idade inválida: {antes - dfCandBem.shape[0]}")
print(dfCandBem.shape)

print((dfCandBem["Total_Gasto_Campanha"] > 0).sum(), "candidatos com gasto > 0")
print(dfCandBem["Total_Gasto_Campanha"].describe())

dfCandBem[
    ["NM_CANDIDATO", "SG_UF", "IDADE", "Total_Bens",
     "Total_Gasto_Campanha", "Total_Gasto_Campanha_Log"]
].sample(min(5, len(dfCandBem)))

Idade mínima elegível para DEPUTADO FEDERAL: 21 anos
Candidatos removidos por idade inválida: 2
(2791, 61)
2024 candidatos com gasto > 0
count    2.791000e+03
mean     2.179525e+05
std      4.610702e+05
min      0.000000e+00
25%      0.000000e+00
50%      2.078000e+04
75%      1.623370e+05
max      3.127763e+06
Name: Total_Gasto_Campanha, dtype: float64


,NM_CANDIDATO,SG_UF,IDADE,Total_Bens,Total_Gasto_Campanha,Total_Gasto_Campanha_Log
428,PAULO ABI-ACKEL,MG,63,20093322.56,1073568.88,13.886500
1248,ROBERTA RODRIGUES DOS REIS MENDES,RJ,45,77396.00,123891.59,11.727170
130,JANE APARECIDA ZANETTI,ES,56,600000.00,0.00,0.000000
1150,GUTEMBERG REIS DE OLIVEIRA,RJ,47,581300.00,2589483.50,14.766969
1251,FELIPE MACEDO BRASILEIRO,RJ,47,227641.27,12800.00,9.457279


> **Nota:** propositalmente **não removemos outliers de patrimônio aqui** — a Fase 1 (análise descritiva) precisa vê-los para serem discutidos, e o tratamento de outliers é específico de cada técnica (vamos fazer isso dentro da Fase 2, clusterização).

## 6) Salvar o dataset limpo

In [ ]:
# Bloco 9: Exportação do dataset consolidado para CSV (com alterações)
nome_uf = UF if UF is not None else 'BRASIL'
nome_cargo = CARGO.replace(' ', '_')
arquivo_saida = f"dados/candidatos_{nome_cargo}_{nome_uf}_{ANO_ELEICAO}.csv"

# [ALTERAÇÃO]: Garantia de exportação do DataFrame dfCandBem atualizado com as colunas Total_Gasto_Campanha e Total_Gasto_Campanha_Log
dfCandBem.to_csv(arquivo_saida, index=False)
print(f"Dataset salvo em: {arquivo_saida}  ({dfCandBem.shape[0]} candidatos, {dfCandBem.shape[1]} colunas)")

Dataset salvo em: dados/candidatos_DEPUTADO_FEDERAL_('SP', 'RJ', 'MG', 'ES')_2026.csv  (2791 candidatos, 61 colunas)


---
## Congelando esta versão

Depois de conferir o resultado acima, faça o commit desse arquivo:

```bash
git add dados/*.csv
git commit -m "Dados: candidatos <CARGO> <UF> <ANO> (snapshot TSE de <DT_GERACAO>)"
```

A partir daí, **esse commit é a versão oficial** que a Fase 1 em diante (e o resto da turma, via `git pull`) vai usar — mesmo que o TSE atualize o arquivo-fonte depois. Só repita a Fase 0 e recommite quando quiser atualizar de propósito.

## Próximo passo

Vá para **`01_analise_descritiva.ipynb`** e use exatamente os mesmos valores de `CARGO`, `UF` e `ANO_ELEICAO` na célula de configuração, para carregar este mesmo arquivo.